<a href="https://colab.research.google.com/github/ssbio/CatRange/blob/fix/colab-python-runtime-stability/CatRange_Inference_Interface.ipynb?flush_cache=true" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open runtime-stability test in Colab"/></a>

**Robust Prediction of Enzyme Variant Kinetics with CatRange**

This notebook predicts enzyme kinetic parameter ranges ($k_{\text{cat}}$ and $K_M$) for protein sequences and substrates using CatRange.

Before the kinetics step runs, the workflow uses [**CLEAN** (An Enzyme Function Prediction Tool using Contrastive Learning)](https://github.com/tttianhao/CLEAN) on each protein sequence to:
- predict a likely **EC number**
- report a **CLEAN confidence score** for the top EC assignment
- decide whether that sequence should continue to the CatRange kinetics step

_No coding experience required. Collapse cells to keep the notebook clean._

---

## **How to Use**
1. **Select Mode**: `Demo`, `Interactive`, `Bulk`, or `Bulk-large`
2. **Select Mutation Pathway**:
   - **Mechanistic Mutation-Aware (default)** - recommended; better reflects graded mutation effects
   - **Binary Over-Simplified (poor and optional)** - legacy simplified mode for comparison only
3. **Run the cells sequentially below**
4. **Enter inputs or upload a CSV when prompted**
5. **Review the EC and kinetics output table**
6. **Download either `inference_results.csv` only or all output files**

> ⚠️ Each time you run the pipeline cell, Colab will:
> - Install required dependencies
> - Download pretrained model files
> - Run CLEAN first, then run CatRange only on rows that pass the CLEAN screen

---

## **What CLEAN Does in This Pipeline**
- CLEAN uses the **protein sequence only** to predict a likely EC number.
- The top EC call is written to `clean_top_ec_number`.
- Its CLEAN confidence score is written to `clean_top_confidence`.
- `clean_top_confidence` is a model confidence score for the top EC call, not an experimental measurement of catalytic activity.
- If a row does **not** pass the CLEAN enzyme screen, that row is still kept in the final output, but the CatRange prediction columns are marked as `skipped`.

## **Modes**
- **`Demo`**: test the pipeline with built-in examples
- **`Interactive`**: manually enter up to 10 enzyme-substrate pairs
- **`Bulk`**: upload a CSV with 10 or fewer rows
  - [Sample CSV](https://drive.google.com/uc?export=download&id=1X9bR67NW-sTNHaKU4W1Htlfxhl-OKMmu)
- **`Bulk-large`**: upload a CSV with more than 10 rows
  - Includes the +/-1 class error columns around each CatRange prediction
  - GPU is recommended for larger jobs
  - If your sequences are long (roughly >500 aa), start with a very small `BATCH_SIZE` such as `1` or `2`, then increase gradually

## **Input Requirements**
- Required CSV columns: `sequence` and `Isomeric SMILES`
- One row should contain one enzyme-substrate pair
- Extra spaces are removed automatically during setup

## **Length Limits**
- Minimum sequence length: **9 amino acids**
- Maximum sequence length: **1022 amino acids**
- Minimum SMILES length: **2 characters**
- Maximum SMILES length: **512 characters**
- Rows outside these limits are kept in the final CSV and marked as `skipped`

## **Output Files**
- `inference_results.csv`: user-facing summary table with EC calls, enzyme classification, kinetics ranges, uncertainty ranges, and units
- `inference_results_detailed.csv`: technical output columns, including the full CLEAN fields and support score
- `clean_screened.csv`: CLEAN results for all submitted rows
- `clean_catrange_ready.csv`: only the rows passed from CLEAN into CatRange
- `run_config.json`: saved settings used for the run


## **Notebook Release**
- `2026-08-28-python-runtime-stability-test-4`


In [ ]:
#@title 1. Prepare input and save the run settings
mode = "Demo"  #@param ["Demo", "Interactive", "Bulk", "Bulk-large"]
mutation_mode = "Mechanistic Mutation-Aware (default)"  #@param ["Mechanistic Mutation-Aware (default)", "Binary Over-Simplified (poor and optional)"]
clean_non_enzyme_threshold = 0.5  #@param {type:"number"}
batch_size = 20  #@param {type:"integer"}

import hashlib
import io
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

NOTEBOOK_RELEASE = "2026-08-28-python-runtime-stability-test-4"
# CATRANGE_FRIENDLY_SETUP_OUTPUT_V1
MAX_SEQ_LENGTH = 1022
MAX_SMILES_LENGTH = 512
CSV_OUT = "infer_input.csv"
RUN_CONFIG_OUT = "run_config.json"
BATCH_SIZE = int(batch_size)

if not 0 <= float(clean_non_enzyme_threshold) <= 1:
    raise ValueError("clean_non_enzyme_threshold must be between 0 and 1.")
if BATCH_SIZE < 1:
    raise ValueError("batch_size must be at least 1.")

try:
    from google.colab import files as _colab_files
    IS_COLAB = True
except Exception:
    _colab_files = None
    IS_COLAB = False

def _clean(df):
    df = df.copy()
    required = {"sequence", "Isomeric SMILES"}
    missing = sorted(required.difference(df.columns))
    if missing:
        raise ValueError(
            "Input CSV is missing required column(s): " + ", ".join(missing)
        )
    df["sequence"] = df["sequence"].astype(str).str.replace(r"\s+", "", regex=True)
    df["Isomeric SMILES"] = df["Isomeric SMILES"].astype(str).str.replace(r"\s+", "", regex=True)
    if "clean_row_id" not in df.columns:
        df.insert(0, "clean_row_id", range(len(df)))
    return df

def _sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _preview(df):
    preview = df.copy()
    preview["sequence_length"] = preview["sequence"].astype(str).str.len()
    preview["smiles_length"] = preview["Isomeric SMILES"].astype(str).str.len()
    preview["within_length_limits"] = (
        preview["sequence_length"].le(MAX_SEQ_LENGTH)
        & preview["smiles_length"].le(MAX_SMILES_LENGTH)
    )
    return preview

if mode == "Bulk":
    if IS_COLAB:
        uploaded = _colab_files.upload()
        if not uploaded:
            raise RuntimeError("No file was uploaded.")
        filename = list(uploaded.keys())[0]
        df = pd.read_csv(io.BytesIO(uploaded[filename]))
    else:
        csv_path = input("Path to CSV with 'sequence' and 'Isomeric SMILES': ").strip()
        if not csv_path:
            raise RuntimeError("No CSV path was provided.")
        df = pd.read_csv(csv_path)
    df = _clean(df)
elif mode == "Bulk-large":
    if IS_COLAB:
        uploaded = _colab_files.upload()
        if not uploaded:
            raise RuntimeError("No file was uploaded.")
        filename = list(uploaded.keys())[0]
        df = pd.read_csv(io.BytesIO(uploaded[filename]))
    else:
        csv_path = input("Path to CSV with 'sequence' and 'Isomeric SMILES': ").strip()
        if not csv_path:
            raise RuntimeError("No CSV path was provided.")
        df = pd.read_csv(csv_path)
    df = _clean(df)
elif mode in ("Interactive", "Demo"):
    pairs = []
    if mode == "Demo":
        pairs = [
            (
                "MKVAVLGAAGGIGQALALLLKTQLPSGSELSLYDIAPVTPGVAVDLSHIPTAVKIKGFSGEDATPALEGADVVLISAGVARKPGMDRSDLFNVNAGIVKNLVQQVAKTCPKACIGIITNPVNTTVAIAAEVLKKAGVYDKNKLFGVTTLDIIRSNTFVAELKGKQPGEVEVPVIGGHSGVTILPLLSQVPGVSFTEQEVADLTKRIQNAGTEVVEAKAGGGSATLSMGQAAARFGLSLVRALQGEQGVVECAYVEGDGQYARFFSQPLLLGKNGVEERKSIGTLSAFEQNALEGMLDTLKKDIALGEEFVNK",
                "C(C(=O)C(=O)O)C(=O)O",
            ),
            (
                "MGVEQILKRKTGVIVGEDVHNLFTYAKEHKFAIPAINVTSSSTAVAALEAARDSKSPIILQTSNGGAAYFAGKGISNEGQNASIKGAIAAAHYIRSIAPAYGIPVVLHSDHCAKKLLPWFDGMLEADEAYFKEHGEPLFSSHMLDLSEETDEENISTCVKYFKRMAAMDQWLEMEIGITGGEEDGVNNENADKEDLYTKPEQVYNVYKALHPISPNFSIAAAFGNCHGLYAGDIALRPEILAEHQKYTREQVGCKEEKPLFLVFHGGSGSTVQEFHTGIDNGVVKVNLDTDCQYAYLTGIRDYVLNKKDYIMSPVGNPEGPEKPNKKFFDPRVWVREGEKTMGAKITKSLETFRTTNTL",
                "C([C@H](C=O)O)OP(=O)(O)O",
            ),
            ("M" * 1023, "C(C=O)"),
            ("MKKVAV", "C" + "C" * 512),
        ]
    else:
        max_pairs = 10
        while len(pairs) < max_pairs:
            sequence = input(f"Sequence #{len(pairs) + 1} (blank to stop): ").strip()
            if not sequence:
                break
            smiles = input("Isomeric SMILES: ").strip()
            if not smiles:
                print("Skipping blank SMILES entry.")
                continue
            pairs.append((sequence, smiles))
    if not pairs:
        raise RuntimeError("No inputs were provided.")
    df = pd.DataFrame(pairs, columns=["sequence", "Isomeric SMILES"])
    df = _clean(df)
else:
    raise ValueError(mode)

preview_df = _preview(df)
df.to_csv(CSV_OUT, index=False)

rows_over_limits = int((~preview_df["within_length_limits"]).sum())
run_config = {
    "notebook_release": NOTEBOOK_RELEASE,
    "mode": mode,
    "mutation_mode": mutation_mode,
    "clean_non_enzyme_threshold": float(clean_non_enzyme_threshold),
    "batch_size": BATCH_SIZE,
    "clean_esm_batches_per_clean_inference": BATCH_SIZE,
    "input_rows": int(len(df)),
    "rows_over_limits": rows_over_limits,
    "input_csv": os.path.abspath(CSV_OUT),
    "input_sha256": _sha256(CSV_OUT),
}
Path(RUN_CONFIG_OUT).write_text(json.dumps(run_config, indent=2))

print(f"Ready: {len(df)} input row(s) | {mode} | {mutation_mode}")
if rows_over_limits:
    print(
        f"Note: {rows_over_limits} row(s) are outside the supported length limits "
        "and will be kept in the results as skipped."
    )

display_columns = [
    column
    for column in [
        "clean_row_id",
        "sequence_length",
        "smiles_length",
        "within_length_limits",
    ]
    if column in preview_df.columns
]
if display_columns:
    display(preview_df[display_columns].head(min(10, len(preview_df))))

env_values = {
    "MODE": mode,
    "CSV_PATH": os.path.abspath(CSV_OUT),
    "BATCH_SIZE": str(BATCH_SIZE),
    "MUTATION_MODE": mutation_mode,
    "CLEAN_NON_ENZYME_THRESHOLD": str(float(clean_non_enzyme_threshold)),
    "RUN_CONFIG_PATH": os.path.abspath(RUN_CONFIG_OUT),
    "NOTEBOOK_RELEASE": NOTEBOOK_RELEASE,
}
os.environ.update(env_values)


In [ ]:
#@title 2. Run CLEAN + CatRange Inference pipeline (EC number + $k_{\text{cat}}$ + $K_M$ <- Mechanistic or Binary, depending on selection above)


# CATRANGE_STREAMED_PIPELINE_V1
import subprocess

pipeline_script = r'''set -euo pipefail
# CATRANGE_FRIENDLY_OUTPUT_V1

export TF_CPP_MIN_LOG_LEVEL=3
export XLA_FLAGS="--xla_cpu_enable_fast_math=false"
export PYTHONWARNINGS="ignore"

if [ -z "${MODE:-}" ]; then
  echo "No mode selected. Run the setup cell first."
  exit 1
fi

: "${CSV_PATH:?}"
: "${BATCH_SIZE:=20}"
: "${CLEAN_NON_ENZYME_THRESHOLD:=0.5}"

if [ -d /content ]; then
  export RUNTIME_ROOT="/content"
else
  export RUNTIME_ROOT="$(pwd)"
fi

run_setup_step() {
  local label="$1"
  local log_path="$2"
  shift 2
  if ! "$@" >"${log_path}" 2>&1; then
    echo "[error] ${label} failed. Last details:"
    tail -n 25 "${log_path}" || true
    exit 1
  fi
}

echo "CatRange run: ${MODE} | ${MUTATION_MODE:-Mechanistic}"

python3 - <<'PY'
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

csv_path = Path(os.environ["CSV_PATH"]).expanduser().resolve()
runtime_root = Path(os.environ["RUNTIME_ROOT"]).resolve()
standalone_clean_script = runtime_root / "standalone_clean_inference.py"
clean_work_dir = runtime_root / ".clean_runtime"
clean_screened_csv = runtime_root / "clean_screened.csv"
clean_catrange_ready_csv = runtime_root / "clean_catrange_ready.csv"
clean_merged_output_csv = runtime_root / "final_inference_results.csv"
clean_env_file = runtime_root / "clean_runtime.env"
clean_non_enzyme_threshold = float(os.environ["CLEAN_NON_ENZYME_THRESHOLD"])

apt_cmd = (
    "apt-get update -qq && "
    "DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends "
    "git curl ca-certificates unzip"
)
# CATRANGE_LOCAL_JUPYTER_SUPPORT_V1
print("[1/3] Preparing the runtime...", flush=True)
required_tools = ("git", "curl", "unzip")
missing_tools = [tool for tool in required_tools if shutil.which(tool) is None]
if missing_tools:
    apt_result = subprocess.run(
        ["bash", "-lc", apt_cmd],
        text=True,
        capture_output=True,
    )
    if apt_result.returncode != 0:
        details = "\n".join(
            part for part in (apt_result.stdout, apt_result.stderr) if part
        ).strip()
        print(
            "[error] Missing local tools: " + ", ".join(missing_tools),
            flush=True,
        )
        if details:
            print("\n".join(details.splitlines()[-25:]), flush=True)
        raise RuntimeError(
            "Install Git, curl, and unzip, then rerun this cell."
        )

standalone_clean_script.write_text('#!/usr/bin/env python3\n"""Standalone CLEAN runner with isolated environment bootstrapping.\n\nScreen mode:\n  - accepts CSV or FASTA input\n  - downloads CLEAN + pretrained weights automatically\n  - runs CLEAN inference in a dedicated venv to avoid esm conflicts\n  - labels each sequence as enzyme/non-enzyme using CLEAN GMM confidence\n  - can write a catrange-ready filtered CSV containing only enzyme rows\n\nMerge mode:\n  - merges a catrange output CSV back into the screened CLEAN CSV by clean_row_id\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport time\nimport zipfile\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Optional, Sequence, Tuple\n\n\nBOOTSTRAP_ENV_VAR = "CLEAN_STANDALONE_BOOTSTRAPPED"\n# CATRANGE_FRIENDLY_CLEAN_OUTPUT_V1\nUV_VERSION = "0.8.14"\nDEFAULT_CLEAN_PYTHON = "3.12"\nCLEAN_REQUIREMENTS = (\'torch==2.4.1\', \'numpy==1.26.4\', \'pandas==2.2.3\', \'scikit-learn==1.5.2\', \'scipy==1.11.4\', \'tqdm==4.66.5\', \'fair-esm==2.0.0\', \'pysam==0.22.1\', \'easydict==1.13\', \'gdown==5.2.0\')\nCLEAN_IMPORTS = (\n    "torch", "numpy", "pandas", "sklearn", "scipy",\n    "tqdm", "esm", "pysam", "easydict", "gdown",\n)\nDEFAULT_PRETRAINED_URL = (\n    "https://drive.google.com/file/d/1kwYd4VtzYuMvJMWXy6Vks91DSUAOcKpZ/view?usp=sharing"\n)\nDEFAULT_REPO_URL = "https://github.com/tttianhao/CLEAN.git"\nDEFAULT_REPO_REF = "f2bf2a4f497fa2cc87dac2a1bb314fee587c0a15"\nREQUIRED_PRETRAINED_FILES = {\n    "100.pt",\n    "70.pt",\n    "split100.pth",\n    "split70.pth",\n    "gmm_ensumble.pkl",\n}\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\n        "--work-dir",\n        default=str(Path.cwd() / ".clean_runtime"),\n        help="Directory for the isolated env, repo clone, downloads, and temp files.",\n    )\n    parser.add_argument(\n        "--clean-repo-dir",\n        default=None,\n        help="Optional existing CLEAN checkout to reuse instead of cloning a new one.",\n    )\n    parser.add_argument(\n        "--repo-url",\n        default=DEFAULT_REPO_URL,\n        help="CLEAN git repository URL used when cloning is required.",\n    )\n    parser.add_argument(\n        "--repo-ref",\n        default=DEFAULT_REPO_REF,\n        help="Pinned CLEAN commit or ref used by the managed checkout.",\n    )\n    parser.add_argument(\n        "--pretrained-url",\n        default=DEFAULT_PRETRAINED_URL,\n        help="Google Drive URL for the pretrained CLEAN assets zip.",\n    )\n    parser.add_argument(\n        "--clean-python",\n        default=DEFAULT_CLEAN_PYTHON,\n        help="Managed Python major.minor used for CLEAN (default: 3.12).",\n    )\n    parser.add_argument(\n        "--train-data",\n        default="split100",\n        choices=["split70", "split100"],\n        help="Which pretrained split to use for CLEAN inference.",\n    )\n    parser.add_argument(\n        "--non-enzyme-threshold",\n        type=float,\n        default=0.5,\n        help=(\n            "Minimum CLEAN top-hit GMM confidence required to call a sequence an enzyme. "\n            "CLEAN does not publish an official non-enzyme cutoff; 0.5 is the default decision boundary."\n        ),\n    )\n    parser.add_argument(\n        "--toks-per-batch",\n        type=int,\n        default=2048,\n        help="Token budget per ESM batch passed through CLEAN_inference.py.",\n    )\n    parser.add_argument(\n        "--esm-batches-per-clean-inference",\n        type=int,\n        default=200,\n        help="How many ESM batches to accumulate before EC inference inside CLEAN.",\n    )\n    parser.add_argument(\n        "--verbose",\n        action="store_true",\n        help="Print extra progress details.",\n    )\n\n    subparsers = parser.add_subparsers(dest="command", required=True)\n\n    screen = subparsers.add_parser("screen", help="Run standalone CLEAN screening + EC prediction.")\n    screen.add_argument("--input-csv", default=None, help="Input CSV containing a sequence column.")\n    screen.add_argument("--input-fasta", default=None, help="Input FASTA file.")\n    screen.add_argument(\n        "--sequence-column",\n        default="sequence",\n        help="Sequence column name when using --input-csv.",\n    )\n    screen.add_argument(\n        "--output-csv",\n        required=True,\n        help="Path to the augmented CSV with CLEAN outputs.",\n    )\n    screen.add_argument(\n        "--job-name",\n        default=None,\n        help="Optional job slug used for intermediate CLEAN filenames.",\n    )\n    screen.add_argument(\n        "--write-catrange-ready-csv",\n        default=None,\n        help="Optional path for a filtered CSV containing only rows marked as enzymes.",\n    )\n\n    merge = subparsers.add_parser("merge", help="Merge catrange results back into screened CLEAN output.")\n    merge.add_argument("--screened-csv", required=True, help="CSV previously created by the screen command.")\n    merge.add_argument(\n        "--catrange-output-csv",\n        required=True,\n        help="CSV emitted by the catrange stage using the filtered catrange-ready CSV.",\n    )\n    merge.add_argument("--output-csv", required=True, help="Path for the final merged CSV.")\n\n    return parser.parse_args()\n\n\ndef log(message: str) -> None:\n    print(message, flush=True)\n\n\ndef run(\n    command: Sequence[str],\n    *,\n    cwd: Optional[Path] = None,\n    env: Optional[Dict[str, str]] = None,\n    capture_output: bool = False,\n    quiet: bool = True,\n) -> subprocess.CompletedProcess[str]:\n    if capture_output:\n        return subprocess.run(\n            list(command),\n            cwd=str(cwd) if cwd else None,\n            env=env,\n            text=True,\n            capture_output=True,\n            check=True,\n        )\n    if not quiet:\n        return subprocess.run(\n            list(command),\n            cwd=str(cwd) if cwd else None,\n            env=env,\n            text=True,\n            check=True,\n        )\n\n    result = subprocess.run(\n        list(command),\n        cwd=str(cwd) if cwd else None,\n        env=env,\n        text=True,\n        capture_output=True,\n    )\n    if result.returncode != 0:\n        combined_output = "\\n".join(part for part in [result.stdout, result.stderr] if part).strip()\n        if combined_output:\n            tail = "\\n".join(combined_output.splitlines()[-40:])\n            log("[error] Last lines from the failed command:")\n            print(tail, flush=True)\n        raise subprocess.CalledProcessError(\n            result.returncode,\n            result.args,\n            output=result.stdout,\n            stderr=result.stderr,\n        )\n    return result\n\n\ndef uv_environment(work_dir: Path) -> Dict[str, str]:\n    env = os.environ.copy()\n    env["UV_CACHE_DIR"] = str(work_dir / "uv-cache")\n    env["UV_PYTHON_INSTALL_DIR"] = str(work_dir / "uv-python")\n    env["UV_MANAGED_PYTHON"] = "1"\n    env["UV_LINK_MODE"] = "copy"\n    return env\n\n\ndef ensure_uv(work_dir: Path) -> Path:\n    # CATRANGE_CLEAN_LINUX_MESSAGE_V1\n    if platform.system().lower() != "linux":\n        raise RuntimeError(\n            "Automatic CLEAN setup currently requires Linux. "\n            "On Windows, run CatRange inside WSL."\n        )\n\n    uv_bin = work_dir / "uv-bin" / "uv"\n    if uv_bin.exists():\n        result = subprocess.run(\n            [str(uv_bin), "--version"],\n            text=True,\n            capture_output=True,\n        )\n        if result.returncode == 0 and result.stdout.strip() == f"uv {UV_VERSION}":\n            return uv_bin\n        uv_bin.unlink()\n\n    uv_bin.parent.mkdir(parents=True, exist_ok=True)\n    machine = platform.machine().lower()\n    targets = {\n        "x86_64": "x86_64-unknown-linux-gnu",\n        "amd64": "x86_64-unknown-linux-gnu",\n        "aarch64": "aarch64-unknown-linux-gnu",\n        "arm64": "aarch64-unknown-linux-gnu",\n    }\n    target = targets.get(machine)\n    if target is None:\n        raise RuntimeError(f"Unsupported Linux architecture for uv: {machine}")\n\n    archive_path = work_dir / f"uv-{UV_VERSION}-{target}.tar.gz"\n    download_path = archive_path.with_suffix(archive_path.suffix + ".download")\n    staged_uv = uv_bin.with_suffix(".download")\n    release_url = (\n        f"https://github.com/astral-sh/uv/releases/download/{UV_VERSION}/"\n        f"uv-{target}.tar.gz"\n    )\n    try:\n        run(\n            [\n                "curl",\n                "-fL",\n                "--retry",\n                "3",\n                "--retry-all-errors",\n                release_url,\n                "-o",\n                str(download_path),\n            ],\n        )\n        download_path.replace(archive_path)\n        with tarfile.open(archive_path, "r:gz") as archive:\n            members = [\n                member\n                for member in archive.getmembers()\n                if member.isfile() and Path(member.name).name == "uv"\n            ]\n            if len(members) != 1:\n                raise RuntimeError(\n                    f"Expected one uv binary in {archive_path}; found {len(members)}."\n                )\n            source = archive.extractfile(members[0])\n            if source is None:\n                raise RuntimeError(f"Could not read uv binary from {archive_path}.")\n            with source, staged_uv.open("wb") as destination:\n                shutil.copyfileobj(source, destination)\n        staged_uv.chmod(0o755)\n        staged_uv.replace(uv_bin)\n    finally:\n        archive_path.unlink(missing_ok=True)\n        download_path.unlink(missing_ok=True)\n        staged_uv.unlink(missing_ok=True)\n\n    result = run([str(uv_bin), "--version"], capture_output=True)\n    if result.stdout.strip() != f"uv {UV_VERSION}":\n        uv_bin.unlink(missing_ok=True)\n        raise RuntimeError(\n            f"Expected uv {UV_VERSION}, received {result.stdout.strip()!r}."\n        )\n    return uv_bin\n\n\ndef environment_fingerprint(python_spec: str) -> str:\n    payload = "\\n".join((UV_VERSION, python_spec, *CLEAN_REQUIREMENTS)).encode()\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef validate_clean_env(python_bin: Path, python_spec: str) -> bool:\n    if not python_bin.exists():\n        return False\n    try:\n        expected_python = tuple(int(part) for part in python_spec.split(".")[:2])\n        if len(expected_python) != 2:\n            return False\n    except ValueError:\n        return False\n\n    expected_versions = dict(item.split("==", 1) for item in CLEAN_REQUIREMENTS)\n    validation_code = (\n        "import importlib,sys\\n"\n        "from importlib.metadata import version\\n"\n        f"assert sys.version_info[:2] == {expected_python!r}, sys.version\\n"\n        f"expected = {expected_versions!r}\\n"\n        "for distribution, expected_version in expected.items():\\n"\n        "    actual = version(distribution)\\n"\n        "    assert actual == expected_version, (distribution, actual, expected_version)\\n"\n        f"for module in {CLEAN_IMPORTS!r}:\\n"\n        "    importlib.import_module(module)\\n"\n    )\n    result = subprocess.run(\n        [str(python_bin), "-c", validation_code],\n        text=True,\n        capture_output=True,\n        timeout=180,\n    )\n    return result.returncode == 0\n\n\ndef acquire_environment_lock(lock_dir: Path, timeout_seconds: int = 900) -> None:\n    deadline = time.monotonic() + timeout_seconds\n    while True:\n        try:\n            lock_dir.mkdir()\n            return\n        except FileExistsError:\n            try:\n                age_seconds = time.time() - lock_dir.stat().st_mtime\n            except FileNotFoundError:\n                continue\n            if age_seconds > timeout_seconds:\n                shutil.rmtree(lock_dir, ignore_errors=True)\n                continue\n            if time.monotonic() >= deadline:\n                raise TimeoutError(f"Timed out waiting for CLEAN environment lock: {lock_dir}")\n            time.sleep(2)\n\n\ndef resolve_work_dir(raw_work_dir: str) -> Path:\n    work_dir = Path(raw_work_dir).expanduser().resolve()\n    work_dir.mkdir(parents=True, exist_ok=True)\n    return work_dir\n\n\ndef resolve_repo_dir(args: argparse.Namespace, work_dir: Path) -> Path:\n    if args.clean_repo_dir:\n        return Path(args.clean_repo_dir).expanduser().resolve()\n    return (work_dir / "CLEAN_repo").resolve()\n\n\ndef ensure_clean_env(args: argparse.Namespace, work_dir: Path) -> Path:\n    env_dir = work_dir / "clean_env"\n    python_bin = env_dir / "bin" / "python"\n    marker_path = env_dir / ".bootstrap_complete"\n    lock_dir = work_dir / ".clean_env.lock"\n    fingerprint = environment_fingerprint(args.clean_python)\n    requirements_path = work_dir / "clean-requirements.txt"\n    requirements_text = "\\n".join(CLEAN_REQUIREMENTS) + "\\n"\n    if not requirements_path.exists() or requirements_path.read_text() != requirements_text:\n        requirements_path.write_text(requirements_text)\n\n    if (\n        marker_path.exists()\n        and marker_path.read_text().strip() == fingerprint\n        and validate_clean_env(python_bin, args.clean_python)\n    ):\n        log("[1/3] Runtime ready (cached).")\n        return python_bin\n\n    acquire_environment_lock(lock_dir)\n    try:\n        if (\n            marker_path.exists()\n            and marker_path.read_text().strip() == fingerprint\n            and validate_clean_env(python_bin, args.clean_python)\n        ):\n            log("[1/3] Runtime ready (cached).")\n            return python_bin\n\n        if env_dir.exists():\n            log("[1/3] Refreshing an incomplete runtime...")\n            shutil.rmtree(env_dir)\n\n        uv_bin = ensure_uv(work_dir)\n        uv_env = uv_environment(work_dir)\n        log("[1/3] Installing runtime dependencies (first run only)...")\n        try:\n            run([str(uv_bin), "python", "install", args.clean_python], env=uv_env)\n            run(\n                [\n                    str(uv_bin),\n                    "venv",\n                    "--managed-python",\n                    "--python",\n                    args.clean_python,\n                    str(env_dir),\n                ],\n                env=uv_env,\n            )\n            run(\n                [\n                    str(uv_bin),\n                    "pip",\n                    "install",\n                    "--python",\n                    str(python_bin),\n                    "-r",\n                    str(requirements_path),\n                ],\n                env=uv_env,\n            )\n            if not validate_clean_env(python_bin, args.clean_python):\n                raise RuntimeError("CLEAN environment validation failed after installation.")\n            marker_path.write_text(fingerprint + "\\n")\n        except Exception:\n            shutil.rmtree(env_dir, ignore_errors=True)\n            raise\n        return python_bin\n    finally:\n        shutil.rmtree(lock_dir, ignore_errors=True)\n\n\ndef ensure_clean_repo(args: argparse.Namespace, repo_dir: Path) -> None:\n    app_dir = repo_dir / "app"\n    entrypoint = app_dir / "CLEAN_inference.py"\n    if args.clean_repo_dir:\n        if not entrypoint.exists():\n            raise FileNotFoundError(\n                f"The provided CLEAN checkout is missing {entrypoint}."\n            )\n        return\n\n    if entrypoint.exists():\n        current_ref = run(\n            ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],\n            capture_output=True,\n        ).stdout.strip()\n        if current_ref == args.repo_ref:\n            return\n        log("      Refreshing the managed CLEAN source files...")\n        run(\n            ["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", args.repo_ref]\n        )\n        run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"])\n        if not entrypoint.exists():\n            raise RuntimeError("The pinned CLEAN checkout is missing app/CLEAN_inference.py.")\n        return\n\n    if repo_dir.exists():\n        shutil.rmtree(repo_dir)\n    repo_dir.parent.mkdir(parents=True, exist_ok=True)\n    log("      Downloading required CLEAN files...")\n    run(["git", "clone", "--depth", "1", args.repo_url, str(repo_dir)])\n    run(\n        ["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", args.repo_ref]\n    )\n    run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"])\n    if not entrypoint.exists():\n        raise RuntimeError("The pinned CLEAN checkout is missing app/CLEAN_inference.py.")\n\n\ndef ensure_pretrained_assets(\n    python_bin: Path,\n    repo_dir: Path,\n    pretrained_url: str,\n    work_dir: Path,\n) -> None:\n    pretrained_dir = repo_dir / "app" / "data" / "pretrained"\n    pretrained_dir.mkdir(parents=True, exist_ok=True)\n    missing = [name for name in REQUIRED_PRETRAINED_FILES if not (pretrained_dir / name).exists()]\n    if not missing:\n        return\n\n    log("      Downloading the CLEAN model...")\n    zip_path = work_dir / "clean_pretrained.zip"\n    run([str(python_bin), "-m", "gdown", "--fuzzy", pretrained_url, "-O", str(zip_path)])\n\n    extract_dir = work_dir / "clean_pretrained_extract"\n    if extract_dir.exists():\n        shutil.rmtree(extract_dir)\n    extract_dir.mkdir(parents=True, exist_ok=True)\n\n    with zipfile.ZipFile(zip_path, "r") as archive:\n        archive.extractall(extract_dir)\n\n    copied = set()\n    for path in extract_dir.rglob("*"):\n        if path.is_file() and path.name in REQUIRED_PRETRAINED_FILES:\n            shutil.copy2(path, pretrained_dir / path.name)\n            copied.add(path.name)\n\n    missing = [name for name in REQUIRED_PRETRAINED_FILES if not (pretrained_dir / name).exists()]\n    if missing:\n        raise RuntimeError(\n            "Downloaded CLEAN assets but could not find all required files: " + ", ".join(sorted(missing))\n        )\n\n\ndef maybe_bootstrap_and_reexec(args: argparse.Namespace) -> None:\n    if os.environ.get(BOOTSTRAP_ENV_VAR) == "1":\n        return\n\n    work_dir = resolve_work_dir(args.work_dir)\n    python_bin = ensure_clean_env(args, work_dir)\n    repo_dir = resolve_repo_dir(args, work_dir)\n    ensure_clean_repo(args, repo_dir)\n    ensure_pretrained_assets(python_bin, repo_dir, args.pretrained_url, work_dir)\n\n    env = os.environ.copy()\n    env[BOOTSTRAP_ENV_VAR] = "1"\n    run([str(python_bin), str(Path(__file__).resolve()), *sys.argv[1:]], env=env, quiet=False)\n    raise SystemExit(0)\n\n\ndef bootstrap_mode_python() -> Path:\n    return Path(sys.executable)\n\n\ndef normalize_sequence(sequence: object) -> str:\n    if sequence is None:\n        return ""\n    sequence = str(sequence).upper()\n    sequence = re.sub(r"\\s+", "", sequence)\n    sequence = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "X", sequence)\n    return sequence\n\n\ndef sanitize_job_name(raw_name: str) -> str:\n    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", raw_name.strip())\n    slug = slug.strip("._-")\n    return slug or "clean_job"\n\n\ndef read_fasta_records(fasta_path: Path) -> List[Tuple[str, str]]:\n    records: List[Tuple[str, str]] = []\n    current_id: Optional[str] = None\n    current_seq: List[str] = []\n    with fasta_path.open() as handle:\n        for line in handle:\n            line = line.strip()\n            if not line:\n                continue\n            if line.startswith(">"):\n                if current_id is not None:\n                    records.append((current_id, "".join(current_seq)))\n                current_id = line[1:].strip() or f"record_{len(records)}"\n                current_seq = []\n            else:\n                current_seq.append(line)\n    if current_id is not None:\n        records.append((current_id, "".join(current_seq)))\n    return records\n\n\ndef load_screen_input(args: argparse.Namespace):\n    import pandas as pd\n\n    if bool(args.input_csv) == bool(args.input_fasta):\n        raise ValueError("Pass exactly one of --input-csv or --input-fasta for the screen command.")\n\n    if args.input_csv:\n        input_csv = Path(args.input_csv).expanduser().resolve()\n        df = pd.read_csv(input_csv)\n        if args.sequence_column not in df.columns:\n            raise ValueError(\n                f"Sequence column \'{args.sequence_column}\' was not found in {input_csv}. "\n                f"Available columns: {list(df.columns)}"\n            )\n        if "clean_row_id" not in df.columns:\n            df.insert(0, "clean_row_id", range(len(df)))\n        input_name = input_csv.stem\n        return df, input_name, "csv"\n\n    input_fasta = Path(args.input_fasta).expanduser().resolve()\n    records = read_fasta_records(input_fasta)\n    rows = [\n        {"clean_row_id": index, "sequence_id": record_id, "sequence": sequence}\n        for index, (record_id, sequence) in enumerate(records)\n    ]\n    df = pd.DataFrame(rows)\n    input_name = input_fasta.stem\n    return df, input_name, "fasta"\n\n\ndef ensure_fasta_index(python_bin: Path, fasta_path: Path) -> None:\n    run(\n        [\n            str(python_bin),\n            "-c",\n            "import sys,pysam; pysam.faidx(sys.argv[1])",\n            str(fasta_path),\n        ]\n    )\n\n\ndef detect_torch_runtime(python_bin: Path) -> str:\n    try:\n        result = run(\n            [\n                str(python_bin),\n                "-c",\n                "import torch; print(\'gpu\' if torch.cuda.is_available() else \'cpu\')",\n            ],\n            capture_output=True,\n        )\n        return result.stdout.strip() or "unknown"\n    except Exception:\n        return "unknown"\n\n\ndef run_clean_inference(\n    args: argparse.Namespace,\n    python_bin: Path,\n    repo_dir: Path,\n    fasta_path: Path,\n    sequence_count: int,\n) -> Path:\n    app_dir = repo_dir / "app"\n    results_dir = app_dir / "results" / "inputs"\n    results_dir.mkdir(parents=True, exist_ok=True)\n    env = os.environ.copy()\n    src_dir = app_dir / "src"\n    env["PYTHONPATH"] = str(src_dir) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")\n\n    ensure_fasta_index(python_bin, fasta_path)\n\n    gmm_path = app_dir / "data" / "pretrained" / "gmm_ensumble.pkl"\n    command = [\n        str(python_bin),\n        "CLEAN_inference.py",\n        "--train_data",\n        args.train_data,\n        "--inference_fasta_folder",\n        str(fasta_path.parent),\n        "--inference_fasta",\n        fasta_path.name,\n        "--inference_fasta_start",\n        "0",\n        "--inference_fasta_end",\n        str(sequence_count),\n        "--toks_per_batch",\n        str(args.toks_per_batch),\n        "--esm_batches_per_clean_inference",\n        str(args.esm_batches_per_clean_inference),\n        "--gmm",\n        str(gmm_path),\n    ]\n    runtime_label = detect_torch_runtime(python_bin)\n    log(\n        f"[2/3] CLEAN: screening {sequence_count} sequence(s) "\n        f"on {runtime_label.upper()}..."\n    )\n    run(command, cwd=app_dir, env=env)\n    fasta_stem = fasta_path.stem\n    return results_dir / f"{fasta_stem}_0_{sequence_count}.csv"\n\n\ndef parse_prediction(raw_prediction: object) -> Tuple[str, List[str], Optional[float]]:\n    if raw_prediction is None:\n        return "", [], None\n    text = str(raw_prediction).strip()\n    if not text or text.lower() == "nan":\n        return "", [], None\n\n    ec_numbers: List[str] = []\n    top_confidence: Optional[float] = None\n    for index, part in enumerate(text.split(";")):\n        part = part.strip()\n        match = re.search(r"EC:([^/\\s]+)\\s*/\\s*([0-9]*\\.?[0-9]+)", part)\n        if not match:\n            continue\n        ec_numbers.append(match.group(1))\n        if index == 0:\n            top_confidence = float(match.group(2))\n\n    top_ec = ec_numbers[0] if ec_numbers else ""\n    return top_ec, ec_numbers, top_confidence\n\n\ndef screen_sequences(args: argparse.Namespace) -> None:\n    import pandas as pd\n\n    work_dir = resolve_work_dir(args.work_dir)\n    repo_dir = resolve_repo_dir(args, work_dir)\n    python_bin = bootstrap_mode_python()\n    ensure_clean_repo(args, repo_dir)\n    ensure_pretrained_assets(python_bin, repo_dir, args.pretrained_url, work_dir)\n\n    df, input_name, input_kind = load_screen_input(args)\n    job_name = sanitize_job_name(args.job_name or input_name)\n    temp_dir = work_dir / "jobs" / job_name\n    temp_dir.mkdir(parents=True, exist_ok=True)\n\n    if "clean_sequence_id" not in df.columns:\n        df["clean_sequence_id"] = ""\n    df["clean_top_confidence"] = None\n    df["clean_non_enzyme_threshold"] = float(args.non_enzyme_threshold)\n    df["clean_is_enzyme"] = False\n    df["clean_should_run_catrange"] = False\n    df["clean_status"] = ""\n    df["clean_top_ec_number"] = ""\n    df["clean_all_ec_numbers"] = ""\n    df["clean_raw_prediction"] = ""\n\n    sequence_records: List[Tuple[int, str, str]] = []\n    if input_kind == "csv":\n        raw_sequences = df[args.sequence_column]\n    else:\n        raw_sequences = df["sequence"]\n\n    for row_index, raw_sequence in enumerate(raw_sequences):\n        clean_row_id = int(df.iloc[row_index]["clean_row_id"])\n        normalized = normalize_sequence(raw_sequence)\n        sequence_id = f"row_{clean_row_id:06d}"\n        df.at[row_index, "clean_sequence_id"] = sequence_id\n        if not normalized:\n            df.at[row_index, "clean_status"] = "empty_sequence"\n            continue\n        sequence_records.append((row_index, sequence_id, normalized))\n\n    output_csv = Path(args.output_csv).expanduser().resolve()\n    output_csv.parent.mkdir(parents=True, exist_ok=True)\n\n    if not sequence_records:\n        log("[result] No non-empty sequences were provided.")\n        df.to_csv(output_csv, index=False)\n        if args.write_catrange_ready_csv:\n            ready_path = Path(args.write_catrange_ready_csv).expanduser().resolve()\n            ready_path.parent.mkdir(parents=True, exist_ok=True)\n            df.iloc[0:0].to_csv(ready_path, index=False)\n        return\n\n    fasta_path = repo_dir / "app" / "data" / "inputs" / f"{job_name}.fasta"\n    fasta_path.parent.mkdir(parents=True, exist_ok=True)\n    with fasta_path.open("w") as handle:\n        for _, sequence_id, normalized in sequence_records:\n            handle.write(f">{sequence_id}\\n{normalized}\\n")\n\n    results_path = run_clean_inference(args, python_bin, repo_dir, fasta_path, len(sequence_records))\n    clean_df = pd.read_csv(results_path)\n    prediction_by_id = {\n        str(row["Seq_ID"]): row.get("Prediction", "")\n        for _, row in clean_df.iterrows()\n    }\n\n    for row_index, sequence_id, _ in sequence_records:\n        raw_prediction = prediction_by_id.get(sequence_id, "")\n        top_ec, all_ecs, top_confidence = parse_prediction(raw_prediction)\n        is_enzyme = bool(top_ec) and top_confidence is not None and top_confidence >= args.non_enzyme_threshold\n\n        df.at[row_index, "clean_top_confidence"] = top_confidence\n        df.at[row_index, "clean_raw_prediction"] = raw_prediction\n        if top_ec:\n            df.at[row_index, "clean_top_ec_number"] = top_ec\n            df.at[row_index, "clean_all_ec_numbers"] = "; ".join(all_ecs)\n        if is_enzyme:\n            df.at[row_index, "clean_is_enzyme"] = True\n            df.at[row_index, "clean_should_run_catrange"] = True\n            df.at[row_index, "clean_status"] = "enzyme"\n        elif top_ec:\n            df.at[row_index, "clean_status"] = "non_enzyme_low_confidence"\n        else:\n            df.at[row_index, "clean_status"] = "no_prediction"\n\n    df.to_csv(output_csv, index=False)\n    enzyme_count = int(df["clean_is_enzyme"].sum())\n    log(\n        "[2/3] CLEAN complete: "\n        f"{enzyme_count} of {len(df)} sequence(s) passed the enzyme screen."\n    )\n\n    if args.write_catrange_ready_csv:\n        ready_path = Path(args.write_catrange_ready_csv).expanduser().resolve()\n        ready_path.parent.mkdir(parents=True, exist_ok=True)\n        ready_df = df[df["clean_should_run_catrange"]].copy()\n        ready_df.to_csv(ready_path, index=False)\n\n\ndef merge_catrange_results(args: argparse.Namespace) -> None:\n    import pandas as pd\n\n    screened_csv = Path(args.screened_csv).expanduser().resolve()\n    catrange_output_csv = Path(args.catrange_output_csv).expanduser().resolve()\n    output_csv = Path(args.output_csv).expanduser().resolve()\n    output_csv.parent.mkdir(parents=True, exist_ok=True)\n\n    screened_df = pd.read_csv(screened_csv)\n    catrange_df = pd.read_csv(catrange_output_csv)\n\n    if "clean_row_id" not in screened_df.columns:\n        raise ValueError(f"{screened_csv} is missing clean_row_id.")\n    if "clean_row_id" not in catrange_df.columns:\n        raise ValueError(\n            f"{catrange_output_csv} is missing clean_row_id. "\n            "Run catrange on the --write-catrange-ready-csv output so that clean_row_id is preserved."\n        )\n\n    candidate_prediction_cols = [\n        column\n        for column in catrange_df.columns\n        if column not in screened_df.columns or column.startswith("Predicted_")\n    ]\n    merged = screened_df.merge(\n        catrange_df[["clean_row_id", *candidate_prediction_cols]],\n        on="clean_row_id",\n        how="left",\n    )\n\n    def prediction_skip_label(row: pd.Series) -> str:\n        status = str(row.get("clean_status", "")).strip().lower()\n        if status == "empty_sequence":\n            return "skipped_empty_sequence"\n        if status == "no_prediction":\n            return "skipped_clean_no_prediction"\n        if status == "non_enzyme_low_confidence":\n            return "skipped_non_enzyme"\n        return "skipped_non_enzyme"\n\n    for column in merged.columns:\n        if column.startswith("Predicted_"):\n            merged[column] = merged[column].where(\n                merged[column].notna(),\n                merged.apply(prediction_skip_label, axis=1),\n            )\n\n    merged.to_csv(output_csv, index=False)\n    log("[done] CLEAN and CatRange results combined.")\n\n\ndef main() -> None:\n    args = parse_args()\n    maybe_bootstrap_and_reexec(args)\n\n    if args.command == "screen":\n        screen_sequences(args)\n    elif args.command == "merge":\n        merge_catrange_results(args)\n    else:\n        raise ValueError(args.command)\n\n\nif __name__ == "__main__":\n    main()\n')
standalone_clean_script.chmod(0o755)

cmd = [
    sys.executable,
    str(standalone_clean_script),
    "--work-dir",
    str(clean_work_dir),
    "--clean-python",
    "3.12",
    "--non-enzyme-threshold",
    str(clean_non_enzyme_threshold),
    "--esm-batches-per-clean-inference",
    os.environ["BATCH_SIZE"],
    "screen",
    "--input-csv",
    str(csv_path),
    "--output-csv",
    str(clean_screened_csv),
    "--write-catrange-ready-csv",
    str(clean_catrange_ready_csv),
    "--job-name",
    "catrange_clean_screen",
]

subprocess.run(cmd, check=True)

screened_df = pd.read_csv(clean_screened_csv)
enzyme_count = int(screened_df["clean_is_enzyme"].sum()) if "clean_is_enzyme" in screened_df.columns else 0

env_lines = {
    "CLEAN_STANDALONE_SCRIPT": str(standalone_clean_script),
    "CLEAN_WORK_DIR": str(clean_work_dir),
    "CLEAN_PYTHON": str(clean_work_dir / "clean_env" / "bin" / "python"),
    "UV_BIN": str(clean_work_dir / "uv-bin" / "uv"),
    "CLEAN_SCREENED_CSV": str(clean_screened_csv),
    "CLEAN_catrange_READY_CSV": str(clean_catrange_ready_csv),
    "CLEAN_MERGED_OUTPUT_CSV": str(clean_merged_output_csv),
    "CLEAN_NO_VALID_EC": "1" if enzyme_count <= 0 else "0",
    "CLEAN_PASSED_COUNT": str(enzyme_count),
}
clean_env_file.write_text(
    "".join(
        f"export {key}={shlex.quote(value)}\n"
        for key, value in env_lines.items()
    )
)
PY

CLEAN_ENV_FILE="${RUNTIME_ROOT}/clean_runtime.env"
if [ -f "${CLEAN_ENV_FILE}" ]; then
  # shellcheck disable=SC1090
  source "${CLEAN_ENV_FILE}"
fi

if [ "${CLEAN_NO_VALID_EC:-0}" = "1" ]; then
  python3 - <<'PY'
import os
from pathlib import Path

import pandas as pd

screened_path = Path(os.environ["CLEAN_SCREENED_CSV"]).expanduser().resolve()
results_path = Path("inference_results.csv").resolve()
df = pd.read_csv(screened_path)

def skip_label(status: object) -> str:
    text = str(status).strip().lower()
    if text == "empty_sequence":
        return "skipped_empty_sequence"
    if text == "no_prediction":
        return "skipped_clean_no_prediction"
    return "skipped_non_enzyme"

labels = df["clean_status"].map(skip_label) if "clean_status" in df.columns else "skipped_non_enzyme"
prediction_columns = [
    "Predicted_Kcat_low",
    "Predicted_Kcat_high",
    "Predicted_Kcat_low (-1 class error)",
    "Predicted_Kcat_high (+1 class error)",
    "Predicted_KM_low",
    "Predicted_KM_high",
    "Predicted_KM_low (-1 class error)",
    "Predicted_KM_high (+1 class error)",
]
for column in prediction_columns:
    df[column] = labels

df.to_csv(results_path, index=False)
print("[done] No rows passed the CLEAN screen; results were saved with skip reasons.")
PY

python3 - <<'PY'
from pathlib import Path

import math
import pandas as pd

results_path = Path("inference_results.csv")
if not results_path.exists():
    raise SystemExit(0)

df = pd.read_csv(results_path)
df.to_csv("inference_results_detailed.csv", index=False)

def as_float(value):
    try:
        if pd.isna(value):
            return None
        if isinstance(value, str) and value.startswith("skipped"):
            return None
        return float(value)
    except Exception:
        return None

def format_number(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return f"{value:.2e}"

def format_range(low, high, unit):
    low_value = as_float(low)
    high_value = as_float(high)
    if low_value is None or high_value is None:
        return ""
    return f"[{format_number(low_value)}, {format_number(high_value)}] {unit}"

def enzyme_screen_result(status):
    text = str(status).strip().lower()
    if text == "enzyme":
        return "Yes"
    if text == "non_enzyme_low_confidence":
        return "No"
    if text == "no_prediction":
        return "No confident EC assignment"
    if text == "empty_sequence":
        return "Sequence missing"
    return "Unavailable"

def pipeline_note(row):
    raw_value = str(row.get("Predicted_Kcat_low", "")).strip()
    if not raw_value or raw_value.lower() == "nan":
        return "Prediction unavailable"
    if raw_value == "skipped_non_enzyme":
        return "catrange was not run because CLEAN did not classify this sequence as an enzyme"
    if raw_value == "skipped_clean_no_prediction":
        return "catrange was not run because CLEAN did not return a confident EC assignment"
    if raw_value == "skipped_empty_sequence":
        return "catrange was not run because the sequence was missing"
    if raw_value.startswith("skipped"):
        return "catrange was not run because the input was outside the supported limits"
    return "catrange prediction completed"

def additional_ec_candidates(row):
    top_ec = str(row.get("clean_top_ec_number", "")).strip()
    all_ec = str(row.get("clean_all_ec_numbers", "")).strip()
    if not all_ec or all_ec.lower() == "nan":
        return ""
    ec_list = [item.strip() for item in all_ec.split(";") if item.strip()]
    filtered = [item for item in ec_list if item != top_ec]
    return "; ".join(filtered)

final_df = pd.DataFrame(
    {
        "Input row": df.get("clean_row_id", pd.Series(range(len(df)))),
        "Protein sequence": df.get("sequence", pd.Series([""] * len(df))),
        "Substrate (Isomeric SMILES)": df.get("Isomeric SMILES", pd.Series([""] * len(df))),
        "Classified as enzyme?": df.get("clean_status", pd.Series([""] * len(df))).map(enzyme_screen_result),
        "Predicted EC number": df.get("clean_top_ec_number", pd.Series([""] * len(df))).fillna(""),
        "Other possible EC numbers": df.apply(additional_ec_candidates, axis=1),
        "Pipeline note": df.apply(pipeline_note, axis=1),
        "Predicted kcat range (s^-1)": [
            format_range(low, high, "s^-1")
            for low, high in zip(
                df.get("Predicted_Kcat_low", pd.Series([""] * len(df))),
                df.get("Predicted_Kcat_high", pd.Series([""] * len(df))),
            )
        ],
        "Predicted kcat uncertainty range (s^-1)": [
            format_range(low, high, "s^-1")
            for low, high in zip(
                df.get("Predicted_Kcat_low (-1 class error)", pd.Series([""] * len(df))),
                df.get("Predicted_Kcat_high (+1 class error)", pd.Series([""] * len(df))),
            )
        ],
        "Predicted KM range (M)": [
            format_range(low, high, "M")
            for low, high in zip(
                df.get("Predicted_KM_low", pd.Series([""] * len(df))),
                df.get("Predicted_KM_high", pd.Series([""] * len(df))),
            )
        ],
        "Predicted KM uncertainty range (M)": [
            format_range(low, high, "M")
            for low, high in zip(
                df.get("Predicted_KM_low (-1 class error)", pd.Series([""] * len(df))),
                df.get("Predicted_KM_high (+1 class error)", pd.Series([""] * len(df))),
            )
        ],
    }
)

final_df.to_csv(results_path, index=False)
print("[done] Results saved: inference_results.csv")
PY
  exit 0
fi

export CSV_PATH="${CLEAN_catrange_READY_CSV}"

# ---------------------------- Stage 2: catrange ----------------------------
# ---------------------------- MECHANISTIC BRANCH (ESM-C, v2-like) ----------------------------
if [ "${MUTATION_MODE}" = "Mechanistic Mutation-Aware (default)" ]; then
  echo "[3/3] CatRange: mechanistic kinetics prediction (ESM-C)..."
  # optional clean-up
  # pip uninstall -y tensorflow tensorflow-gpu tensorflow-cpu tensorflow-intel tf-nightly 2>/dev/null || true
  : "${UV_BIN:?CLEAN bootstrap did not provide UV_BIN}"
  CATRANGE_RUNTIME_DIR="${RUNTIME_ROOT}/.catrange_runtime"
  MECH_ENV="${CATRANGE_RUNTIME_DIR}/mechanistic-py312"
  MECH_PY="${MECH_ENV}/bin/python"
  MECH_REQUIREMENTS="${CATRANGE_RUNTIME_DIR}/mechanistic-py312-requirements.txt"
  MECH_MARKER="${MECH_ENV}/.bootstrap_complete"
  mkdir -p "${CATRANGE_RUNTIME_DIR}"
  printf '%s\n' \
    'numpy==2.1.3' \
    'pandas==2.2.2' \
    'scikit-learn==1.7.1' \
    'joblib==1.2.0' \
    'xgboost==2.1.4' \
    'torch==2.4.0' \
    'torchvision==0.19.0' \
    'torchaudio==2.4.0' \
    'tqdm==4.66.5' \
    'esm==3.2.3' \
    'httpx==0.28.1' \
    'biotite==1.2.0' \
    'transformers==4.48.1' \
    'huggingface_hub==0.25.2' > "${MECH_REQUIREMENTS}"
  MECH_FINGERPRINT="py312-uv0.8.14-$(sha256sum "${MECH_REQUIREMENTS}" | cut -d' ' -f1)"
  export UV_CACHE_DIR="${CATRANGE_RUNTIME_DIR}/uv-cache"
  export UV_PYTHON_INSTALL_DIR="${CLEAN_WORK_DIR}/uv-python"
  export UV_MANAGED_PYTHON=1
  export UV_LINK_MODE=copy
  MECH_VALID=0
  if [ -x "${MECH_PY}" ] && [ -f "${MECH_MARKER}" ] && [ "$(tr -d '\r\n' < "${MECH_MARKER}")" = "${MECH_FINGERPRINT}" ]; then
    if "${MECH_PY}" -c 'import sys,torch,numpy,pandas,sklearn,joblib,xgboost,tqdm,esm,httpx,biotite,transformers,huggingface_hub; assert sys.version_info[:2] == (3, 12)' >/dev/null 2>&1; then
      MECH_VALID=1
    fi
  fi
  if [ "${MECH_VALID}" = "1" ]; then
    echo "      CatRange runtime ready (cached)."
  else
    echo "      Installing CatRange dependencies (first run only)..."
    rm -rf "${MECH_ENV}"
    run_setup_step "Python 3.12 setup" /tmp/catrange_mech_setup.log "${UV_BIN}" python install 3.12
    run_setup_step "CatRange environment setup" /tmp/catrange_mech_setup.log "${UV_BIN}" venv --managed-python --python 3.12 "${MECH_ENV}"
    run_setup_step "CatRange dependency setup" /tmp/catrange_mech_setup.log "${UV_BIN}" pip install --python "${MECH_PY}" -r "${MECH_REQUIREMENTS}"
    run_setup_step "CatRange environment check" /tmp/catrange_mech_setup.log "${MECH_PY}" -c 'import sys,torch,numpy,pandas,sklearn,joblib,xgboost,tqdm,esm,httpx,biotite,transformers,huggingface_hub; assert sys.version_info[:2] == (3, 12)'
    printf '%s\n' "${MECH_FINGERPRINT}" > "${MECH_MARKER}"
  fi
  : > latex_queue.txt
  if ! MODE="$MODE" CSV_PATH="$CSV_PATH" BATCH_SIZE="$BATCH_SIZE" stdbuf -oL "${MECH_PY}" - <<'PY' 2>/tmp/catrange_mechanistic_stderr.log
import os, sys, warnings, torch, argparse, random, math, joblib, numpy as np, pandas as pd, io, gc, contextlib
warnings.filterwarnings("ignore")
import logging
logging.disable(logging.CRITICAL)
from torch.utils.data import Dataset, DataLoader
from torch.serialization import add_safe_globals
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from huggingface_hub import snapshot_download
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
# ---------------- basic runtime ----------------
os.environ["OMP_NUM_THREADS"]="1"; os.environ["MKL_NUM_THREADS"]="1"
try:
    torch.set_num_threads(1)
except Exception:
    pass
MODE = os.environ.get("MODE","Demo").strip()
CSV_PATH = os.environ["CSV_PATH"]
BATCH_SIZE = int(os.environ.get("BATCH_SIZE","20"))
ENCODER = os.environ.get("ENCODER","esmc").strip().lower()          # only "esmc" supported here
ALLOW_ENCODER_MISMATCH = int(os.environ.get("ALLOW_ENCODER_MISMATCH","1"))
try:
    from google.colab import files as _cf
    IS_COLAB=True
except Exception:
    _cf=None; IS_COLAB=False
def check_device(): return torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = check_device()
print(f"      Computing predictions on {str(device).upper()}..."); sys.stdout.flush()
MAX_SEQ_LENGTH=1022
MAX_SMILES_LENGTH=512
MIN_SEQ_LENGTH=9
MIN_SMILES_LENGTH=2
# ---------------- class ranges ----------------
#  Build class ranges for kcat and km
kcat_log_bins = np.array([0, 1e-08, 1e-02, 1e-01, 1e+00, 1e+01, 1e+02, 1e+03, 1e+08])
km_log_bins   = np.array([1e-14, 1e-05, 1e-04, 1e-03, 1e-02, 1e-01, 1e+04])
class_ranges_kcat = {i: {"low": float(kcat_log_bins[i]), "high": float(kcat_log_bins[i+1])}
                for i in range(len(kcat_log_bins)-1)}
class_ranges_km = {i: {"low": float(km_log_bins[i]), "high": float(km_log_bins[i+1])}
                for i in range(len(km_log_bins)-1)}
def format_sci(v):
    s=f"{v:.2e}"
    if "e" in s:
        b,e=s.split("e"); return f"{b}x10^{int(e)}"
    return s
warnings.filterwarnings("ignore"); add_safe_globals([argparse.Namespace])
# ---------------- data load & basic cleaning ----------------
df=pd.read_csv(CSV_PATH)
df["sequence"]=df["sequence"].astype(str).str.replace(r"\s+","",regex=True).str.upper() \
            .str.replace(r"[^ACDEFGHIKLMNPQRSTVWY]", "X", regex=True)
df["Isomeric SMILES"]=df["Isomeric SMILES"].astype(str).str.replace(r"\s+","",regex=True)
pairs=list(zip(df["sequence"],df["Isomeric SMILES"]))
# ---------------- ESM-C loader & embed ----------------
def load_esmc_model(device, work_dir="."):
    """
    Loads ESM-C 600M to work_dir/esmc_model/... if needed, then returns eval() model.
    """
    repo_id = "EvolutionaryScale/esmc-600m-2024-12"
    local_dir = os.path.join(work_dir, "esmc_model")
    weights_rel = "data/weights/esmc_600m_2024_12_v0.pth"
    weights_path = os.path.join(local_dir, weights_rel)
    from huggingface_hub import logging as hf_logging
    hf_logging.set_verbosity_error()
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    silent_output = io.StringIO()
    if not os.path.exists(weights_path):
        with contextlib.redirect_stdout(silent_output), contextlib.redirect_stderr(silent_output):
            snapshot_download(
                repo_id=repo_id,
                local_dir=local_dir,
                force_download=False,
                local_dir_use_symlinks=False,
                tqdm_class=None,
            )
    with contextlib.redirect_stdout(silent_output), contextlib.redirect_stderr(silent_output):
        model = ESMC.from_pretrained("esmc_600m").to(device)
        if os.path.exists(weights_path):
            try:
                sd = torch.load(weights_path, map_location=device)
                model.load_state_dict(sd, strict=False)
            except Exception:
                pass
    model.eval()
    return model
@torch.no_grad()
def esmc_mean_embed(model, sequence: str, device):
    """
    Returns mean-pooled embedding over residues (excl. special tokens).
    """
    protein = ESMProtein(sequence=sequence)
    toks = model.encode(protein)  # (B, L)
    out = model.logits(toks, LogitsConfig(sequence=True, structure=True, return_embeddings=True))
    L = len(sequence)
    reps = out.embeddings[0, 1:L-1]  # drop [CLS]/[EOS]-like
    return reps.mean(dim=0).float().to(device)  # (D_esmc,)
# ---------------- chem encoder (unchanged) ----------------
_ESMC=None; _TOK=None; _CHEM=None
def _get_models():
    global _ESMC,_TOK,_CHEM
    if ENCODER != "esmc":
        raise RuntimeError("This cell implements only ENCODER=esmc. (esm2 path omitted.)")
    if _ESMC is None:
        _ESMC = load_esmc_model(device)
    if _TOK is None or _CHEM is None:
        _TOK=AutoTokenizer.from_pretrained('seyonec/PubChem10M_SMILES_BPE_450k')
        _CHEM=AutoModel.from_pretrained('seyonec/PubChem10M_SMILES_BPE_450k'); _CHEM.eval().to(device)
    return _ESMC, _TOK, _CHEM
# ---------------- dataset/standardization (unchanged split) ----------------
class TensorDataset(Dataset):
    def __init__(self,d,l): self.d,self.l=d,l
    def __len__(self): return len(self.d)
    def __getitem__(self,i): return self.d[i],self.l[i]
def dataset_to_tensors(ds):
    ld=DataLoader(ds,batch_size=len(ds),shuffle=False); return next(iter(ld))
def standardize_x_global_separate(data,g1,s1,g2,s2):
    # legacy split: first 1280 dims are sequence, remainder is chem
    # X1,X2=data[:,:1280],data[:,1280:]          # ESM-2
    X1,X2 = data[:, :1152], data[:, 1152:]       # ESM-C
    s1=torch.clamp(s1,min=1e-7); s2=torch.clamp(s2,min=1e-7)
    return torch.cat(((X1-g1)/s1,(X2-g2)/s2),dim=1).squeeze(1)
class StandardizedDatasetGlobalSeparate(Dataset):
    def __init__(self,sub,g1,s1,g2,s2): self.sub=sub; self.g1=g1; self.s1=s1; self.g2=g2; self.s2=s2
    def __len__(self): return len(self.sub)
    def __getitem__(self,i):
        x,y=self.sub[i]
        if len(x.shape)==1: x=x.unsqueeze(1)
        return standardize_x_global_separate(x,self.g1,self.s1,self.g2,self.s2),y
def apply_global_standardization_separate(ds,g1,s1,g2,s2):
    return StandardizedDatasetGlobalSeparate(ds,g1,s1,g2,s2)
# legacy global stats for sequence (1152) and chem block
global_mean_1=torch.tensor(-0.0004980484955012798,device=device)
global_std_1=torch.tensor(0.027508573606610298,device=device)
global_mean_2=torch.tensor(-2.7928688723477535e-05,device=device)
global_std_2=torch.tensor(0.6221200823783875,device=device)
# ---------------- model weights (unchanged) ----------------
kcat_model_v1b_path="model_weights/kcat_model_v1b.pkl"
km_model_v1b_path="model_weights/km_model_v1b.pkl"
if not (os.path.exists(kcat_model_v1b_path) and os.path.exists(km_model_v1b_path)):
    os.system("wget -q https://huggingface.co/ssbio/CatRange/resolve/main/model_weights.zip -O model_weights.zip")
    os.system("unzip -o -q model_weights.zip")
# ---------------- inference wrapper ----------------
class KcatInference:
    def __init__(self,model_path,device=None,verbose=True):
        self.device=device if device else device
        self.model=joblib.load(model_path); self.verbose=verbose
        self.X_test_tensor=None; self.y1_test_tensor=None; self.keep_local=[]
    def load_data_from_pairs(self,pairs):
        embs=[]; keep=[]
        esmc_model, tok, chem = _get_models()
        if self.verbose:
            print(f"Models loaded (ENCODER={ENCODER}). Embedding sequences and substrates..."); sys.stdout.flush()
        for i,(seq,smi) in enumerate(pairs,1):
            if not isinstance(seq,str) or not isinstance(smi,str): continue
            if len(seq)>MAX_SEQ_LENGTH or len(seq)<MIN_SEQ_LENGTH: continue
            # ---- sequence (ESM-C) ----
            try:
                es = esmc_mean_embed(esmc_model, seq, device)   # (D_esmc,)
                # enforce legacy 1152-dim for downstream scaler/models
                target_d = 1152
                d = es.numel()
                if d != target_d:
                    if not ALLOW_ENCODER_MISMATCH:
                        raise RuntimeError(
                            f"ESM-C embedding dim {d} != {target_d} expected by trained models. "
                            f"Set ALLOW_ENCODER_MISMATCH=1 to pad/truncate (for plumbing only), "
                            f"or retrain catrange with ESM-C."
                        )
                    es = es[:target_d] if d > target_d else torch.cat([es, torch.zeros(target_d - d, device=es.device)])
            except Exception:
                continue
            # ---- substrate SMILES (PubChem tokenizer/encoder) ----
            try:
                inp=tok([smi],return_tensors='pt',padding=True,truncation=False)
                inp={k:v.to(device) for k,v in inp.items()}
                if inp['input_ids'].shape[1]>MAX_SMILES_LENGTH or inp['input_ids'].shape[1]<MIN_SMILES_LENGTH: continue
                with torch.no_grad(): out=chem(**inp)
                cs=out.last_hidden_state.mean(dim=1).squeeze(0).float()
            except Exception:
                continue
            if es.dim()>1: es=es.flatten()
            if cs.dim()>1: cs=cs.flatten()
            embs.append(torch.cat((es,cs))); keep.append(i-1)
        if not embs:
            self.X_test_tensor=None; self.y1_test_tensor=None; self.keep_local=[]
            return
        self.X_test_tensor=torch.stack(embs).to(device)
        self.y1_test_tensor=torch.zeros(len(embs),dtype=torch.long).to(device)
        self.keep_local=keep
    def standardize_test_data(self,g1,s1,g2,s2):
        self.test_dataset_std=apply_global_standardization_separate(
            TensorDataset(self.X_test_tensor,self.y1_test_tensor),g1,s1,g2,s2
        ) if self.X_test_tensor is not None else None
    def convert_to_numpy(self):
        if self.test_dataset_std is None: return None,None,[]
        X,y=dataset_to_tensors(self.test_dataset_std); return X.cpu().numpy(),y.cpu().numpy(),self.keep_local
    def predict(self,X): return self.model.predict(X)
    def display_prediction_ranges_kcat(self,preds,cr):
        for i,p in enumerate(preds):
            if p is None or p=="skipped": print(f"Sample {i+1}: skipped due to excessive length")
            # else: print(f"Sample {i+1}: Predicted Class = {p}, kcat range = [{format_sci(cr[p]['low'])}, {format_sci(cr[p]['high'])}]")
            else: print(f"Sample {i+1}: Predicted kcat range = [{format_sci(cr[p]['low'])}, {format_sci(cr[p]['high'])}]")
    def display_prediction_ranges_km(self,preds,cr):
        for i,p in enumerate(preds):
            if p is None or p=="skipped": print(f"Sample {i+1}: skipped due to excessive length")
            # else: print(f"Sample {i+1}: Predicted Class = {p}, km range = [{format_sci(cr[p]['low'])}, {format_sci(cr[p]['high'])}]")
            else: print(f"Sample {i+1}: Predicted km range = [{format_sci(cr[p]['low'])}, {format_sci(cr[p]['high'])}]")
# ---------------- driver ----------------
pairs_all=pairs
valid=[i for i,(s,m) in enumerate(pairs_all)
      if isinstance(s,str) and isinstance(m,str)
      and len(s)<=MAX_SEQ_LENGTH and len(m)<=MAX_SMILES_LENGTH and len(s)>=MIN_SEQ_LENGTH and len(m)>=MIN_SMILES_LENGTH]
if not valid:
    df["Predicted_Kcat_low"]="skipped"; df["Predicted_Kcat_high"]="skipped"
    df["Predicted_KM_low"]="skipped"; df["Predicted_KM_high"]="skipped"
    df.to_csv("inference_results.csv",index=False)
    print("[3/3] CatRange complete.")
    if MODE in ("Bulk","Bulk-large") and IS_COLAB: _cf.download("inference_results.csv")
    sys.exit(0)
def run_batch(idxs):
    b=[pairs_all[i] for i in idxs]
    inf=KcatInference(model_path=kcat_model_v1b_path,device=device,verbose=False)
    inf.load_data_from_pairs(b)
    if not inf.keep_local:
        return [],[],[]
    inf.standardize_test_data(global_mean_1,global_std_1,global_mean_2,global_std_2)
    X,_,keep_local=inf.convert_to_numpy()
    yk=KcatInference(model_path=kcat_model_v1b_path,device=device,verbose=False).predict(X)
    ym=KcatInference(model_path=km_model_v1b_path,device=device,verbose=False).predict(X)
    del inf, X; gc.collect()
    return yk,ym,keep_local
N=len(df)
kcat_low_full=['skipped']*N; kcat_high_full=['skipped']*N; kcat_low_m1=['skipped']*N; kcat_high_p1=['skipped']*N
km_low_full=['skipped']*N; km_high_full=['skipped']*N; km_low_m1=['skipped']*N; km_high_p1=['skipped']*N
max_kcat_c=max(class_ranges_kcat.keys()); max_km_c=max(class_ranges_km.keys())
if MODE=="Bulk-large":
    total_batches = math.ceil(len(valid)/BATCH_SIZE)
    for b in range(0, len(valid), BATCH_SIZE):
        idxs=valid[b:b+BATCH_SIZE]; yk,ym,keep=run_batch(idxs)
        if not keep: continue
        for j,k in enumerate(keep):
            i0=idxs[k]; ck=int(yk[j]); cm=int(ym[j])
            kcat_low_full[i0]=class_ranges_kcat[ck]["low"]; kcat_high_full[i0]=class_ranges_kcat[ck]["high"]
            kcat_low_m1[i0]=class_ranges_kcat[max(ck-1,0)]["low"]; kcat_high_p1[i0]=class_ranges_kcat[min(ck+1,max_kcat_c)]["high"]
            km_low_full[i0]=class_ranges_km[cm]["low"]; km_high_full[i0]=class_ranges_km[cm]["high"]
            km_low_m1[i0]=class_ranges_km[max(cm-1,0)]["low"]; km_high_p1[i0]=class_ranges_km[min(cm+1,max_km_c)]["high"]
        gc.collect(); sys.stdout.flush()
else:
    yk,ym,keep=run_batch(valid)
    if keep:
        for j,k in enumerate(keep):
            i0=valid[k]; ck=int(yk[j]); cm=int(ym[j])
            kcat_low_full[i0]=class_ranges_kcat[ck]["low"]; kcat_high_full[i0]=class_ranges_kcat[ck]["high"]
            kcat_low_m1[i0]=class_ranges_kcat[max(ck-1,0)]["low"]; kcat_high_p1[i0]=class_ranges_kcat[min(ck+1,max_kcat_c)]["high"]
            km_low_full[i0]=class_ranges_km[cm]["low"]; km_high_full[i0]=class_ranges_km[cm]["high"]
            km_low_m1[i0]=class_ranges_km[max(cm-1,0)]["low"]; km_high_p1[i0]=class_ranges_km[min(cm+1,max_km_c)]["high"]
df['Predicted_Kcat_low']=kcat_low_full
df['Predicted_Kcat_high']=kcat_high_full
df['Predicted_Kcat_low (-1 class error)']=kcat_low_m1
df['Predicted_Kcat_high (+1 class error)']=kcat_high_p1
df['Predicted_KM_low']=km_low_full
df['Predicted_KM_high']=km_high_full
df['Predicted_KM_low (-1 class error)']=km_low_m1
df['Predicted_KM_high (+1 class error)']=km_high_p1

df.to_csv("inference_results.csv",index=False)
print("[3/3] CatRange complete."); sys.stdout.flush()
# try:
#     if MODE=="Bulk-large" and IS_COLAB:
#         _cf.download("inference_results.csv")
#     if MODE=="Bulk" and IS_COLAB and os.environ.get("DOWNLOAD_CSV","")=="1":
#         _cf.download("inference_results.csv")
# except Exception as e:
#     print(f"[warn] Auto-download failed in subprocess: {e}")
PY
  then
    echo "[error] CatRange inference failed. Last details:"
    tail -n 25 /tmp/catrange_mechanistic_stderr.log || true
    exit 1
  fi
# ---------------------------- BINARY BRANCH (ESM-2, v1-like) ----------------------------
else
  echo "[3/3] CatRange: binary kinetics prediction (ESM-2)..."
  : "${UV_BIN:?CLEAN bootstrap did not provide UV_BIN}"
  CATRANGE_RUNTIME_DIR="${RUNTIME_ROOT}/.catrange_runtime"
  BINARY_ENV="${CATRANGE_RUNTIME_DIR}/binary-py310"
  BINARY_PY="${BINARY_ENV}/bin/python"
  BINARY_REQUIREMENTS="${CATRANGE_RUNTIME_DIR}/binary-py310-requirements.txt"
  BINARY_MARKER="${BINARY_ENV}/.bootstrap_complete"
  mkdir -p "${CATRANGE_RUNTIME_DIR}"
  printf '%s\n' \
    'numpy==1.23.5' \
    'pandas==1.5.3' \
    'scikit-learn==1.1.3' \
    'imbalanced-learn==0.8.1' \
    'joblib==1.2.0' \
    'xgboost==2.1.4' \
    'torch==2.4.0' \
    'torchvision==0.19.0' \
    'torchaudio==2.4.0' \
    'transformers==4.33.3' \
    'fair-esm==2.0.0' \
    'mkl==2022.1.0' \
    'mkl-service==2.4.0' \
    'intel-openmp==2022.1.0' \
    'tqdm==4.66.5' > "${BINARY_REQUIREMENTS}"
  BINARY_FINGERPRINT="py310-uv0.8.14-$(sha256sum "${BINARY_REQUIREMENTS}" | cut -d' ' -f1)"
  export UV_CACHE_DIR="${CATRANGE_RUNTIME_DIR}/uv-cache"
  export UV_PYTHON_INSTALL_DIR="${CLEAN_WORK_DIR}/uv-python"
  export UV_MANAGED_PYTHON=1
  export UV_LINK_MODE=copy
  BINARY_VALID=0
  if [ -x "${BINARY_PY}" ] && [ -f "${BINARY_MARKER}" ] && [ "$(tr -d '\r\n' < "${BINARY_MARKER}")" = "${BINARY_FINGERPRINT}" ]; then
    if "${BINARY_PY}" -c 'import sys,torch,numpy,pandas,sklearn,imblearn,joblib,xgboost,transformers,esm,tqdm; assert sys.version_info[:2] == (3, 10)' >/dev/null 2>&1; then
      BINARY_VALID=1
    fi
  fi
  if [ "${BINARY_VALID}" = "1" ]; then
    echo "      CatRange runtime ready (cached)."
  else
    echo "      Installing CatRange dependencies (first run only)..."
    rm -rf "${BINARY_ENV}"
    run_setup_step "Python 3.10 setup" /tmp/catrange_binary_setup.log "${UV_BIN}" python install 3.10
    run_setup_step "CatRange environment setup" /tmp/catrange_binary_setup.log "${UV_BIN}" venv --managed-python --python 3.10 "${BINARY_ENV}"
    run_setup_step "CatRange dependency setup" /tmp/catrange_binary_setup.log "${UV_BIN}" pip install --python "${BINARY_PY}" -r "${BINARY_REQUIREMENTS}"
    run_setup_step "CatRange environment check" /tmp/catrange_binary_setup.log "${BINARY_PY}" -c 'import sys,torch,numpy,pandas,sklearn,imblearn,joblib,xgboost,transformers,esm,tqdm; assert sys.version_info[:2] == (3, 10)'
    printf '%s\n' "${BINARY_FINGERPRINT}" > "${BINARY_MARKER}"
  fi
  : > latex_queue.txt
  if ! MODE="$MODE" CSV_PATH="$CSV_PATH" BATCH_SIZE="$BATCH_SIZE" stdbuf -oL "${BINARY_PY}" - <<'PY' 2>/tmp/catrange_binary_stderr.log
import os, sys, warnings, torch, argparse, random, math, joblib, numpy as np, pandas as pd, io, gc
from torch.utils.data import Dataset, DataLoader
from torch.serialization import add_safe_globals
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import esm
os.environ["OMP_NUM_THREADS"]="1"; os.environ["MKL_NUM_THREADS"]="1"
try:
    torch.set_num_threads(1)
except Exception:
    pass
MODE=os.environ.get("MODE","Demo").strip()
CSV_PATH=os.environ["CSV_PATH"]
BATCH_SIZE=int(os.environ.get("BATCH_SIZE","20"))
try:
    from google.colab import files as _cf
    IS_COLAB=True
except Exception:
    _cf=None; IS_COLAB=False
def check_device(): return torch.device("cuda" if torch.cuda.is_available() else "cpu")
device=check_device()
print(f"      Computing predictions on {str(device).upper()}..."); sys.stdout.flush()
MAX_SEQ_LENGTH=1022
MAX_SMILES_LENGTH=512
MIN_SEQ_LENGTH=9
MIN_SMILES_LENGTH=2

class_ranges_kcat={0:{"low":0.0,"high":3.32e-8},1:{"low":3.33e-8,"high":1.0e-2},2:{"low":1.01e-2,"high":1.0e-1},3:{"low":1.01e-1,"high":1.0},4:{"low":1.001,"high":10.0},5:{"low":1.004e1,"high":1.0e2},6:{"low":1.0025e2,"high":1.0e3},7:{"low":1.002e3,"high":7.0e7}}
class_ranges_km={0:{"low":1.0e-10,"high":1.0e-5},1:{"low":1.01e-5,"high":1.0e-4},2:{"low":1.002e-4,"high":1.0e-3},3:{"low":1.002e-3,"high":1.0e-2},4:{"low":1.008e-2,"high":1.0e-1},5:{"low":1.01e-1,"high":1.02e2}}
def format_sci(v):
    s=f"{v:.2e}"
    if "e" in s:
        b,e=s.split("e"); return f"{b}x10^{int(e)}"
    return s
warnings.filterwarnings("ignore"); add_safe_globals([argparse.Namespace])
df=pd.read_csv(CSV_PATH)
df["sequence"]=df["sequence"].astype(str).str.replace(r"\s+","",regex=True).str.upper().str.replace(r"[^ACDEFGHIKLMNPQRSTVWY]", "X", regex=True)
df["Isomeric SMILES"]=df["Isomeric SMILES"].astype(str).str.replace(r"\s+","",regex=True)
pairs=list(zip(df["sequence"],df["Isomeric SMILES"]))
def load_esm2_model(device,work_dir=".",verbose=True):
    url="https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt"
    fn=url.split("/")[-1]; p=os.path.join(work_dir,fn)
    if not os.path.exists(p):
        m=torch.hub.load_state_dict_from_url(url,progress=True,map_location=device); torch.save(m,p)
    else:
        if verbose: print("ESM2 model weights found locally. Loading from disk...")
        m=torch.load(p,map_location=device)
    model,alphabet=esm.pretrained.load_model_and_alphabet_core("esm2_t33_650M_UR50D",m)
    model.eval().to(device); return model,alphabet
class TensorDataset(Dataset):
    def __init__(self,d,l): self.d,self.l=d,l
    def __len__(self): return len(self.d)
    def __getitem__(self,i): return self.d[i],self.l[i]
def dataset_to_tensors(ds):
    ld=DataLoader(ds,batch_size=len(ds),shuffle=False); return next(iter(ld))
def standardize_x_global_separate(data,g1,s1,g2,s2):
    X1,X2=data[:,:1280],data[:,1280:]; s1=torch.clamp(s1,min=1e-7); s2=torch.clamp(s2,min=1e-7)
    return torch.cat(((X1-g1)/s1,(X2-g2)/s2),dim=1).squeeze(1)
class StandardizedDatasetGlobalSeparate(Dataset):
    def __init__(self,sub,g1,s1,g2,s2): self.sub=sub; self.g1=g1; self.s1=s1; self.g2=g2; self.s2=s2
    def __len__(self): return len(self.sub)
    def __getitem__(self,i):
        x,y=self.sub[i]
        if len(x.shape)==1: x=x.unsqueeze(1)
        return standardize_x_global_separate(x,self.g1,self.s1,self.g2,self.s2),y
def apply_global_standardization_separate(ds,g1,s1,g2,s2):
    return StandardizedDatasetGlobalSeparate(ds,g1,s1,g2,s2)
global_mean_1=torch.tensor(-0.0006011285004206002,device=device)
global_std_1=torch.tensor(0.18902993202209473,device=device)
global_mean_2=torch.tensor(-0.00015002528380136937,device=device)
global_std_2=torch.tensor(0.6113553047180176,device=device)
kcat_model_path="model_weights/kcat_model.pkl"
km_model_path="model_weights/km_model.pkl"
if not (os.path.exists(kcat_model_path) and os.path.exists(km_model_path)):
    os.system("wget -q https://huggingface.co/ssbio/CatRange/resolve/main/model_weights.zip -O model_weights.zip")
    os.system("unzip -o -q model_weights.zip")
_ESM=None; _ALPH=None; _TOK=None; _CHEM=None
def _get_models():
    global _ESM,_ALPH,_TOK,_CHEM
    if _ESM is None or _ALPH is None:
        _ESM,_ALPH=load_esm2_model(device,verbose=False)
    if _TOK is None or _CHEM is None:
        _TOK=AutoTokenizer.from_pretrained('seyonec/PubChem10M_SMILES_BPE_450k')
        _CHEM=AutoModel.from_pretrained('seyonec/PubChem10M_SMILES_BPE_450k'); _CHEM.eval().to(device)
    return _ESM,_ALPH,_TOK,_CHEM
class KcatInference:
    def __init__(self,model_path,device=None,verbose=True):
        self.device=device if device else device
        self.model=joblib.load(model_path); self.verbose=verbose
        self.X_test_tensor=None; self.y1_test_tensor=None; self.keep_local=[]
    def load_data_from_pairs(self,pairs):
        embs=[]; keep=[]
        esm_model,alphabet,tok,chem=_get_models(); bc=alphabet.get_batch_converter()
        if self.verbose: print("Models loaded. Embedding sequences and substrates..."); sys.stdout.flush()
        for i,(seq,smi) in enumerate(pairs,1):
            if not isinstance(seq,str) or not isinstance(smi,str): continue
            if len(seq)>MAX_SEQ_LENGTH or len(seq)<MIN_SEQ_LENGTH: continue
            try:
                _,_,bt=bc([(f"sample_{i}",seq)]); bt=bt.to(device); bl=(bt!=alphabet.padding_idx).sum(1)
                with torch.no_grad(): r=esm_model(bt,repr_layers=[33],return_contacts=False)
                es=r["representations"][33][0,1:bl-1].mean(dim=0).float()
            except Exception:
                continue
            try:
                inp=tok([smi],return_tensors='pt',padding=True,truncation=False); inp={k:v.to(device) for k,v in inp.items()}
                if inp['input_ids'].shape[1]>MAX_SMILES_LENGTH or inp['input_ids'].shape[1]<MIN_SMILES_LENGTH: continue
                with torch.no_grad(): out=chem(**inp)
                cs=out.last_hidden_state.mean(dim=1).squeeze(0).float()
            except Exception:
                continue
            if es.dim()>1: es=es.flatten()
            if cs.dim()>1: cs=cs.flatten()
            embs.append(torch.cat((es,cs))); keep.append(i-1)
        if not embs:
            self.X_test_tensor=None; self.y1_test_tensor=None; self.keep_local=[]
            return
        self.X_test_tensor=torch.stack(embs).to(device); self.y1_test_tensor=torch.zeros(len(embs),dtype=torch.long).to(device); self.keep_local=keep
    def standardize_test_data(self,g1,s1,g2,s2):
        self.test_dataset_std=apply_global_standardization_separate(TensorDataset(self.X_test_tensor,self.y1_test_tensor),g1,s1,g2,s2) if self.X_test_tensor is not None else None
    def convert_to_numpy(self):
        if self.test_dataset_std is None: return None,None,[]
        X,y=dataset_to_tensors(self.test_dataset_std); return X.cpu().numpy(),y.cpu().numpy(),self.keep_local
    def predict(self,X): return self.model.predict(X)
    def display_prediction_ranges_kcat(self,preds,cr):
        for i,p in enumerate(preds):
            if p is None or p=="skipped": print(f"Sample {i+1}: skipped due to excessive length")
            else: print(f"Sample {i+1}: Predicted Class = {p}, kcat range = [{format_sci(cr[p]['low'])}, {format_sci(cr[p]['high'])}]")
    def display_prediction_ranges_km(self,preds,cr):
        for i,p in enumerate(preds):
            if p is None or p=="skipped": print(f"Sample {i+1}: skipped due to excessive length")
            else: print(f"Sample {i+1}: Predicted Class = {p}, km range = [{format_sci(cr[p]['low'])}, {format_sci(cr[p]['high'])}]")
pairs_all=pairs
valid=[i for i,(s,m) in enumerate(pairs_all) if isinstance(s,str) and isinstance(m,str) and len(s)<=MAX_SEQ_LENGTH and len(m)<=MAX_SMILES_LENGTH]
if not valid:
    df["Predicted_Kcat_low"]="skipped"; df["Predicted_Kcat_high"]="skipped"; df["Predicted_KM_low"]="skipped"; df["Predicted_KM_high"]="skipped"
    df.to_csv("inference_results.csv",index=False); print("[3/3] CatRange complete.")
    if MODE in ("Bulk","Bulk-large") and IS_COLAB: _cf.download("inference_results.csv")
    sys.exit(0)
def run_batch(idxs):
    b=[pairs_all[i] for i in idxs]
    inf=KcatInference(model_path=kcat_model_path,device=device,verbose=False)
    inf.load_data_from_pairs(b)
    if not inf.keep_local:
        return [],[],[]
    inf.standardize_test_data(global_mean_1,global_std_1,global_mean_2,global_std_2)
    X,_,keep_local=inf.convert_to_numpy()
    yk=KcatInference(model_path=kcat_model_path,device=device,verbose=False).predict(X)
    ym=KcatInference(model_path=km_model_path,device=device,verbose=False).predict(X)
    del inf, X; gc.collect()
    return yk,ym,keep_local
N=len(df)
kcat_low_full=['skipped']*N; kcat_high_full=['skipped']*N; kcat_low_m1=['skipped']*N; kcat_high_p1=['skipped']*N
km_low_full=['skipped']*N; km_high_full=['skipped']*N; km_low_m1=['skipped']*N; km_high_p1=['skipped']*N
max_kcat_c=max(class_ranges_kcat.keys()); max_km_c=max(class_ranges_km.keys())
if MODE=="Bulk-large":
    total_batches = math.ceil(len(valid)/BATCH_SIZE)
    for b in range(0, len(valid), BATCH_SIZE):
        idxs=valid[b:b+BATCH_SIZE]; yk,ym,keep=run_batch(idxs)
        if not keep:
            continue
        for j,k in enumerate(keep):
            i0=idxs[k]; ck=int(yk[j]); cm=int(ym[j])
            kcat_low_full[i0]=class_ranges_kcat[ck]["low"]; kcat_high_full[i0]=class_ranges_kcat[ck]["high"]
            kcat_low_m1[i0]=class_ranges_kcat[max(ck-1,0)]["low"]; kcat_high_p1[i0]=class_ranges_kcat[min(ck+1,max_kcat_c)]["high"]
            km_low_full[i0]=class_ranges_km[cm]["low"]; km_high_full[i0]=class_ranges_km[cm]["high"]
            km_low_m1[i0]=class_ranges_km[max(cm-1,0)]["low"]; km_high_p1[i0]=class_ranges_km[min(cm+1,max_km_c)]["high"]
        gc.collect(); sys.stdout.flush()
else:
    yk,ym,keep=run_batch(valid)
    if keep:
        for j,k in enumerate(keep):
            i0=valid[k]; ck=int(yk[j]); cm=int(ym[j])
            kcat_low_full[i0]=class_ranges_kcat[ck]["low"]; kcat_high_full[i0]=class_ranges_kcat[ck]["high"]
            kcat_low_m1[i0]=class_ranges_kcat[max(ck-1,0)]["low"]; kcat_high_p1[i0]=class_ranges_kcat[min(ck+1,max_kcat_c)]["high"]
            km_low_full[i0]=class_ranges_km[cm]["low"]; km_high_full[i0]=class_ranges_km[cm]["high"]
            km_low_m1[i0]=class_ranges_km[max(cm-1,0)]["low"]; km_high_p1[i0]=class_ranges_km[min(cm+1,max_km_c)]["high"]
df['Predicted_Kcat_low']=kcat_low_full
df['Predicted_Kcat_high']=kcat_high_full
df['Predicted_Kcat_low (-1 class error)']=kcat_low_m1
df['Predicted_Kcat_high (+1 class error)']=kcat_high_p1
df['Predicted_KM_low']=km_low_full
df['Predicted_KM_high']=km_high_full
df['Predicted_KM_low (-1 class error)']=km_low_m1
df['Predicted_KM_high (+1 class error)']=km_high_p1
df.to_csv("inference_results.csv",index=False)
print("[3/3] CatRange complete."); sys.stdout.flush()
PY
    then
      echo "[error] CatRange inference failed. Last details:"
      tail -n 25 /tmp/catrange_binary_stderr.log || true
      exit 1
    fi
fi

if [ -n "${CLEAN_SCREENED_CSV:-}" ] && [ -f "${CLEAN_SCREENED_CSV}" ] && [ -n "${CLEAN_STANDALONE_SCRIPT:-}" ] && [ -f "${CLEAN_STANDALONE_SCRIPT}" ]; then
  CLEAN_STANDALONE_BOOTSTRAPPED=1 "${CLEAN_PYTHON}" "${CLEAN_STANDALONE_SCRIPT}" \
    --work-dir "${CLEAN_WORK_DIR:-./.clean_runtime}" \
    merge \
    --screened-csv "${CLEAN_SCREENED_CSV}" \
    --catrange-output-csv "inference_results.csv" \
    --output-csv "${CLEAN_MERGED_OUTPUT_CSV:-final_inference_results.csv}"
  mv "${CLEAN_MERGED_OUTPUT_CSV:-final_inference_results.csv}" "inference_results.csv"
fi

python3 - <<'PY'
from pathlib import Path

import math
import pandas as pd

results_path = Path("inference_results.csv")
if not results_path.exists():
    raise SystemExit(0)

df = pd.read_csv(results_path)
df.to_csv("inference_results_detailed.csv", index=False)

def as_float(value):
    try:
        if pd.isna(value):
            return None
        if isinstance(value, str) and value.startswith("skipped"):
            return None
        return float(value)
    except Exception:
        return None

def format_number(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return f"{value:.2e}"

def format_range(low, high, unit):
    low_value = as_float(low)
    high_value = as_float(high)
    if low_value is None or high_value is None:
        return ""
    return f"[{format_number(low_value)}, {format_number(high_value)}] {unit}"

def enzyme_screen_result(status):
    text = str(status).strip().lower()
    if text == "enzyme":
        return "Yes"
    if text == "non_enzyme_low_confidence":
        return "No"
    if text == "no_prediction":
        return "No confident EC assignment"
    if text == "empty_sequence":
        return "Sequence missing"
    return "Unavailable"

def pipeline_note(row):
    raw_value = str(row.get("Predicted_Kcat_low", "")).strip()
    if not raw_value or raw_value.lower() == "nan":
        return "Prediction unavailable"
    if raw_value == "skipped_non_enzyme":
        return "catrange was not run because CLEAN did not classify this sequence as an enzyme"
    if raw_value == "skipped_clean_no_prediction":
        return "catrange was not run because CLEAN did not return a confident EC assignment"
    if raw_value == "skipped_empty_sequence":
        return "catrange was not run because the sequence was missing"
    if raw_value.startswith("skipped"):
        return "catrange was not run because the input was outside the supported limits"
    return "catrange prediction completed"

def additional_ec_candidates(row):
    top_ec = str(row.get("clean_top_ec_number", "")).strip()
    all_ec = str(row.get("clean_all_ec_numbers", "")).strip()
    if not all_ec or all_ec.lower() == "nan":
        return ""
    ec_list = [item.strip() for item in all_ec.split(";") if item.strip()]
    filtered = [item for item in ec_list if item != top_ec]
    return "; ".join(filtered)


# final_df = pd.DataFrame(
#     {
#         "Input row": df.get("clean_row_id", pd.Series(range(len(df)))),
#         "Protein sequence": df.get("sequence", pd.Series([""] * len(df))),
#         "Substrate (Isomeric SMILES)": df.get("Isomeric SMILES", pd.Series([""] * len(df))),
#         "Classified as enzyme?": df.get("clean_status", pd.Series([""] * len(df))).map(enzyme_screen_result),
#         "Predicted EC number": df.get("clean_top_ec_number", pd.Series([""] * len(df))).fillna(""),
#         "Other possible EC numbers": df.apply(additional_ec_candidates, axis=1),
#         "Pipeline note": df.apply(pipeline_note, axis=1),
#         "Predicted kcat range (s^-1)": [
#             format_range(low, high, "s^-1")
#             for low, high in zip(
#                 df.get("Predicted_Kcat_low", pd.Series([""] * len(df))),
#                 df.get("Predicted_Kcat_high", pd.Series([""] * len(df))),
#             )
#         ],
#         "Predicted kcat uncertainty range (s^-1)": [
#             format_range(low, high, "s^-1")
#             for low, high in zip(
#                 df.get("Predicted_Kcat_low (-1 class error)", pd.Series([""] * len(df))),
#                 df.get("Predicted_Kcat_high (+1 class error)", pd.Series([""] * len(df))),
#             )
#         ],
#         "Predicted KM range (M)": [
#             format_range(low, high, "M")
#             for low, high in zip(
#                 df.get("Predicted_KM_low", pd.Series([""] * len(df))),
#                 df.get("Predicted_KM_high", pd.Series([""] * len(df))),
#             )
#         ],
#         "Predicted KM uncertainty range (M)": [
#             format_range(low, high, "M")
#             for low, high in zip(
#                 df.get("Predicted_KM_low (-1 class error)", pd.Series([""] * len(df))),
#                 df.get("Predicted_KM_high (+1 class error)", pd.Series([""] * len(df))),
#             )
#         ],
#     }
# )

# Keep ALL original/input columns first
final_df = df.copy()

# Add user-facing summary columns without dropping existing ones
final_df["Input row"] = df.get("clean_row_id", pd.Series(range(len(df))))
final_df["Protein sequence"] = df.get("sequence", pd.Series([""] * len(df)))
final_df["Substrate (Isomeric SMILES)"] = df.get("Isomeric SMILES", pd.Series([""] * len(df)))
final_df["Classified as enzyme?"] = df.get("clean_status", pd.Series([""] * len(df))).map(enzyme_screen_result)
final_df["Predicted EC number"] = df.get("clean_top_ec_number", pd.Series([""] * len(df))).fillna("")
final_df["Other possible EC numbers"] = df.apply(additional_ec_candidates, axis=1)
final_df["Pipeline note"] = df.apply(pipeline_note, axis=1)

final_df["Predicted kcat range (s^-1)"] = [
    format_range(low, high, "s^-1")
    for low, high in zip(
        df.get("Predicted_Kcat_low", pd.Series([""] * len(df))),
        df.get("Predicted_Kcat_high", pd.Series([""] * len(df))),
    )
]

final_df["Predicted kcat uncertainty range (s^-1)"] = [
    format_range(low, high, "s^-1")
    for low, high in zip(
        df.get("Predicted_Kcat_low (-1 class error)", pd.Series([""] * len(df))),
        df.get("Predicted_Kcat_high (+1 class error)", pd.Series([""] * len(df))),
    )
]

final_df["Predicted KM range (M)"] = [
    format_range(low, high, "M")
    for low, high in zip(
        df.get("Predicted_KM_low", pd.Series([""] * len(df))),
        df.get("Predicted_KM_high", pd.Series([""] * len(df))),
    )
]

final_df["Predicted KM uncertainty range (M)"] = [
    format_range(low, high, "M")
    for low, high in zip(
        df.get("Predicted_KM_low (-1 class error)", pd.Series([""] * len(df))),
        df.get("Predicted_KM_high (+1 class error)", pd.Series([""] * len(df))),
    )
]

final_df.to_csv(results_path, index=False)
print("[done] Results saved: inference_results.csv")
PY
'''

process = subprocess.Popen(
    ["bash", "-lc", pipeline_script],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
if process.stdout is None:
    raise RuntimeError("Could not read CatRange workflow output.")
for output_line in process.stdout:
    print(output_line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        "CatRange workflow stopped. See the concise [error] details above."
    )


In [ ]:
#@title 3. Review the EC and kinetics results

from pathlib import Path

import pandas as pd
from IPython.display import display

results_path = Path("inference_results.csv")
if not results_path.exists():
    raise FileNotFoundError("inference_results.csv was not found. Run the pipeline cell first.")

df = pd.read_csv(results_path)
# CATRANGE_FRIENDLY_REVIEW_OUTPUT_V1
enzyme_count = int(df.get("Classified as enzyme?", pd.Series(dtype=str)).eq("Yes").sum())
prediction_count = int(
    df.get("Pipeline note", pd.Series(dtype=str)).eq("catrange prediction completed").sum()
)
skipped_count = len(df) - prediction_count

summary = pd.DataFrame(
    [
        {
            "Input rows": len(df),
            "Passed CLEAN": enzyme_count,
            "CatRange predictions": prediction_count,
            "Skipped": skipped_count,
        }
    ]
)
print("Run complete")
display(summary)

preview_columns = [
    column
    for column in [
        "Input row",
        "Classified as enzyme?",
        "Predicted EC number",
        "Pipeline note",
        "Predicted kcat range (s^-1)",
        "Predicted KM range (M)",
    ]
    if column in df.columns
]

if preview_columns:
    display(df.loc[:, preview_columns].head(min(10, len(df))))
else:
    display(df.head(min(10, len(df))))

print("Saved: inference_results.csv")


In [ ]:
#@title 4. Download results
download_choice = "Results only (inference_results.csv)"  #@param ["Results only (inference_results.csv)", "All output files (zip)"]

from pathlib import Path
import zipfile

runtime_root = Path("/content") if Path("/content").exists() else Path.cwd()
results_path = runtime_root / "inference_results.csv"
bundle_path = runtime_root / "catrange_results_bundle.zip"
candidate_files = [
    results_path,
    runtime_root / "inference_results_detailed.csv",
    runtime_root / "infer_input.csv",
    runtime_root / "run_config.json",
    runtime_root / "clean_screened.csv",
    runtime_root / "clean_catrange_ready.csv",
]

if not results_path.exists():
    raise FileNotFoundError("inference_results.csv was not found. Run the earlier cells first.")

existing_files = [path for path in candidate_files if path.exists()]
download_path = results_path

if download_choice == "All output files (zip)":
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in existing_files:
            archive.write(path, arcname=path.name)
    print("Created zip file with:")
    for path in existing_files:
        print(f"- {path.name}")
    print(f"\nSaved {bundle_path.name}")
    download_path = bundle_path
else:
    print(f"Ready to download {results_path.name}")

try:
    from google.colab import files
    files.download(str(download_path))
except Exception:
    print(f"Download is only automatic in Colab. The file is available at {download_path}.")
